In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [7]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Reshape
from tensorflow.keras.optimizers import Adam

def get_model():
    model = Sequential()
    model.add(Dense(100, activation='elu', input_shape=(1,)))
    model.add(Dense(100, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(0.01), loss='mean_squared_error')
    return model

In [8]:
from tensorflow.keras.callbacks import Callback

class SavePrediction(Callback):
    """
    Callbacks which stores predicted
    labels in history at each epoch.
    """
    def __init__(self):
        self.X = np.linspace(-0.7, 0.6, 100).reshape(-1, 1)
        self.custom_history_ = []
        super().__init__()

    def on_epoch_end(self, batch, logs={}):
        """Applied at the end of each epoch"""
        predictions = self.model.predict_on_batch(self.X).ravel()
        self.custom_history_.append(predictions)

In [9]:
predictor_columns = ['pzabovezmean', 'pzabove2', 'zq5', 'zq10',
    'zq15', 'zq20', 'zq25', 'zq30', 'zq35', 'zq40', 'zq45', 'zq50', 'zq55',
    'zq60', 'zq65', 'zq70', 'zq75', 'zq80', 'zq85', 'zq90', 'zq95',
    'zpcum1', 'zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7',
    'zpcum8', 'zpcum9'
    ]

target_column = 'Dgv'

data_sweden = pd.read_csv(r'datasets/rs_sweden.csv', index_col=[0])

#create source, target datasets
#evaluate and rain on latvia instead (keep naming for simplicity)
data_latvia = pd.read_csv(r'datasets/rs_lettland.csv', index_col=[0])
data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
data_train, data_temp = train_test_split(data_latvia, test_size=0.1, random_state=1)
data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=1)

#"General" base dataset (to use for transfer)
Xs = np.array(data_sweden[predictor_columns])
ys = np.array(data_sweden[target_column])

#Specific train and test set
Xt_lab = np.array(data_train[predictor_columns])
yt_lab = np.array(data_train[target_column])

X_target_val = np.array(data_val[predictor_columns])
y_target_val = np.array(data_val[target_column])

X_target_test = np.array(data_test[predictor_columns])
y_target_test = np.array(data_test[target_column])

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_29828\528591093.py:10: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'datasets/rs_sweden.csv', index_col=[0])


In [10]:
from adapt.instance_based import TrAdaBoostR2

model = TrAdaBoostR2(get_model(), n_estimators=30, random_state=0)

save_preds = SavePrediction()
model.fit(Xs, ys, Xt_lab, yt_lab,
          callbacks=[save_preds], epochs=100, batch_size=110, verbose=0)

c:\Users\Dag Bjornberg\AI\transfertreeboost\env\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Dag Bjornberg\AI\transfertreeboost\env\Lib\site-packages\adapt\instance_based\_tradaboost.py:541: SyntaxWarning: invalid escape sequence '\c'
  max_{\\epsilon \\in \\epsilon_S \cup \\epsilon_T} \\epsilon`.
c:\Users\Dag Bjornberg\AI\transfertreeboost\env\Lib\site-packages\adapt\instance_based\_tradaboost.py:700: SyntaxWarning: invalid escape sequence '\c'
  max_{\\epsilon \\in \\epsilon_S \cup \\epsilon_T} \\epsilon`.


AttributeError: 'Sequential' object has no attribute '_is_compiled'